# CFP_Moirai_Forecasts

Generates 1-step-ahead VaR forecasts using Salesforce Moirai 2.0 Small (11.4M) via mixture distribution sampling. 1000 samples per day.

**Paper:** Pele, Lessmann, Hardle (2026)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path('../cfp_ijf_data')
RET_DIR = DATA_DIR / 'returns'

ASSETS = [
    'SP500','STOXX','GDAXI','CACT','FTSE100','ICLN','NIKKEI','HSI','BOVESPA','NIFTY','ASX200',
    'CBU0','TLT','IBGL','DJCI','GOLD','WTI','NATGAS','BTC','ETH','EURUSD','GBPUSD','USDJPY','AUDUSD'
]
ALPHAS = [0.01, 0.025, 0.05, 0.10]
CONTEXT = 512
N_SAMPLES = 1000

def load_returns(asset):
    df = pd.read_csv(RET_DIR / f'{asset}.csv', parse_dates=['date']).set_index('date').sort_index()
    df = df[df['log_return'].abs() <= 0.50]
    return df['log_return']

print(f'Assets: {len(ASSETS)}, Context: {CONTEXT}, Samples: {N_SAMPLES}')

MODELS = [('moirai2', 'Salesforce/moirai-1.1-R-small', 'Moirai-2.0')]

In [ ]:
import torch
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
from gluonts.dataset.pandas import PandasDataset
from gluonts.torch.model.predictor import PyTorchPredictor

model = MoiraiForecast.load_from_checkpoint(
    prediction_length=1, context_length=CONTEXT, patch_size='auto', num_samples=N_SAMPLES,
    target_dim=1, feat_dynamic_real_dim=0, past_feat_dynamic_real_dim=0,
    checkpoint_path='Salesforce/moirai-1.1-R-small',
)

out_dir = DATA_DIR / 'moirai2'
out_dir.mkdir(exist_ok=True)

for asset in tqdm(ASSETS, desc='Moirai-2.0'):
    ret = load_returns(asset)
    n = len(ret)
    vals = ret.values
    dates = ret.index
    records = []
    for t in range(CONTEXT, n):
        context_df = pd.DataFrame({'target': vals[t-CONTEXT:t]},
                                   index=pd.period_range(dates[t-CONTEXT], periods=CONTEXT, freq='D'))
        ds = PandasDataset(dict(context_df), target='target', freq='D')
        predictor = model.create_predictor(batch_size=1)
        forecasts = list(predictor.predict(ds))
        samples = forecasts[0].samples.flatten()
        row = {'date': dates[t]}
        for alpha in ALPHAS:
            row[f'VaR_{alpha}'] = np.percentile(samples, alpha * 100)
        row['mean'] = samples.mean()
        row['std'] = samples.std()
        records.append(row)
    df_out = pd.DataFrame(records).set_index('date')
    df_out.to_parquet(out_dir / f'{asset}.parquet')
print(f'Moirai-2.0: {len(ASSETS)} assets saved')

In [ ]:
# Verify outputs
from pathlib import Path
for model_slug, _, label in MODELS:
    out_dir = DATA_DIR / model_slug
    files = list(out_dir.glob('*.parquet'))
    print(f'{label}: {len(files)} parquet files')